# SynSteg: полный нейросетевой пересчёт на GPU (FP32)

Этот ноутбук запускает **новый пересчёт сохранённых текстов** из приватного репозитория. Он не генерирует новый закрытый тест и не открывает prospective-данные. Основной корпус: **1949 записей**; расширенный корпус качества: **3449 записей**. Это не утверждение о пересчёте всех 13460 записей расширенного архива.

В Colab выберите **Runtime → Change runtime type → GPU**. Нужен Python 3.12. Целевой runtime — **GPU класса 16 ГиБ**, при достаточной свободной памяти. Только веса пары Qwen в FP32 занимают примерно 12,3 ГБ (11,5 ГиБ); дополнительно нужна память для активаций и временных буферов. Достаточность 16 ГиБ не гарантируется: важны реально свободная видеопамять и RAM хоста. При нехватке памяти сохраните частичные результаты; **автоматического перехода на BF16/CPU нет**.

Скачайте ZIP актуальной ветки приватного репозитория через GitHub **Code → Download ZIP**. Далее загрузите его в ноутбук. Токен GitHub не требуется и не должен вставляться в код. Не публикуйте этот ноутбук с загруженными приватными данными или результатами.

Запускайте ячейки по порядку. Последняя ячейка позволяет скачать и полные, и частичные результаты. После потери runtime локальные файлы Colab исчезают: скачайте архив до закрытия сессии.


In [ ]:
from pathlib import Path, PurePosixPath
import os, sys, json, zipfile, stat, tempfile, shutil
from google.colab import files

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f"Нужен Python 3.12, текущий: {sys.version.split()[0]}. Выберите совместимый runtime.")

if 'REPO_ROOT' in globals() and Path(REPO_ROOT).is_dir():
    print("Используется уже загруженный репозиторий:", REPO_ROOT)
else:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Загрузите один ZIP репозитория.")
    upload_name, upload_bytes = next(iter(uploaded.items()))
    if not upload_name.lower().endswith('.zip'):
        raise ValueError("Ожидается ZIP репозитория.")
    stage = Path(tempfile.mkdtemp(prefix='synsteg-repository-', dir='/content'))
    archive_path = stage / 'repository.zip'
    archive_path.write_bytes(upload_bytes)
    extracted = stage / 'extracted'
    extracted.mkdir()
    try:
        with zipfile.ZipFile(archive_path) as archive:
            members = archive.infolist()
            if len(members) > 20000 or sum(x.file_size for x in members) > 2 * 1024**3:
                raise ValueError("Архив превышает лимит 20000 файлов / 2 ГиБ распакованных данных.")
            seen = set()
            for member in members:
                path = PurePosixPath(member.filename)
                if (path.is_absolute() or '..' in path.parts or '\\' in member.filename
                        or not path.parts or ':' in path.parts[0]
                        or stat.S_ISLNK(member.external_attr >> 16)
                        or member.flag_bits & 1):
                    raise ValueError("Небезопасная, символьная или зашифрованная запись ZIP.")
                canonical = str(path)
                if canonical in seen:
                    raise ValueError("ZIP содержит повторяющиеся пути.")
                seen.add(canonical)
                destination = extracted.joinpath(*path.parts)
                if not destination.resolve().is_relative_to(extracted.resolve()):
                    raise ValueError("Выход за каталог распаковки.")
                if member.is_dir():
                    destination.mkdir(parents=True, exist_ok=True)
                else:
                    destination.parent.mkdir(parents=True, exist_ok=True)
                    with archive.open(member) as source, destination.open('xb') as target:
                        shutil.copyfileobj(source, target)
        roots = [p.parent.parent for p in extracted.rglob('scripts/import_source.py')]
        if len(roots) != 1:
            raise ValueError("Не найден единственный корень репозитория SynSteg.")
        REPO_ROOT = roots[0].resolve()
        required = ['scripts/import_source.py', 'scripts/build_neural_corpus.py',
                    'scripts/cache_neural_models.py', 'scripts/run_neural_suite.py',
                    'configs/neural/model_revisions.json', 'requirements.txt', 'requirements-neural.txt']
        missing = [p for p in required if not (REPO_ROOT / p).is_file()]
        if missing:
            raise ValueError("ZIP содержит неполную или старую версию проекта: " + ', '.join(missing))
    except BaseException:
        shutil.rmtree(stage, ignore_errors=True)
        raise
    finally:
        del uploaded, upload_bytes

os.chdir(REPO_ROOT)
os.environ['HF_HOME'] = '/content/synsteg-hf-cache'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTHONUNBUFFERED'] = '1'
RUN_OUTPUT = REPO_ROOT / 'results/neural-suite-colab-fp32'
LOG_DIRECTORY = REPO_ROOT / 'results/colab-setup-logs'
LOG_DIRECTORY.mkdir(parents=True, exist_ok=True)
print("Репозиторий:", REPO_ROOT)
print("Результаты:", RUN_OUTPUT)
print("Кэш публичных моделей:", os.environ['HF_HOME'])


## Журналы и остановка

Следующая функция сохраняет вывод каждой команды и каждые 15 секунд показывает последние строки журнала. При остановке ячейки она завершает всю группу дочерних процессов: сначала SIGTERM, затем при необходимости SIGKILL. Частичные оценки остаются на диске; повторный запуск использует проверенные контрольные точки при неизменных коде, конфигурации и среде.


In [ ]:
import subprocess, signal, time, shlex

def tail_log(path, maximum_bytes=12000, lines=12):
    path = Path(path)
    if not path.exists():
        return ''
    with path.open('rb') as handle:
        handle.seek(max(0, path.stat().st_size - maximum_bytes))
        data = handle.read().decode('utf-8', errors='replace')
    return '\n'.join(data.splitlines()[-lines:])

def terminate_group(process):
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        pass
    try:
        process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        pass
    # The group can outlive its leader; kill remaining grandchildren too.
    try:
        os.killpg(process.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    process.wait(timeout=15)

def run_logged(arguments, log_name):
    log_path = LOG_DIRECTORY / log_name
    print('Запуск:', shlex.join([str(x) for x in arguments]), flush=True)
    started = time.monotonic()
    with log_path.open('a', encoding='utf-8') as log:
        log.write('\n--- command: ' + shlex.join([str(x) for x in arguments]) + '\n')
        log.flush()
        process = subprocess.Popen([str(x) for x in arguments], cwd=REPO_ROOT,
                                   env=os.environ.copy(), stdout=log, stderr=subprocess.STDOUT,
                                   text=True, start_new_session=True)
        try:
            while process.poll() is None:
                try:
                    process.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    print(f'Работа продолжается: {(time.monotonic()-started)/60:.1f} мин; PID {process.pid}', flush=True)
                    print(tail_log(log_path), flush=True)
            returncode = process.returncode
        except BaseException:
            terminate_group(process)
            print('Дочерние процессы остановлены. Скачайте частичные результаты последней ячейкой.', flush=True)
            raise
    print(tail_log(log_path), flush=True)
    if returncode != 0:
        raise RuntimeError(f'Команда завершилась с кодом {returncode}; журнал: {log_path}')
    return log_path


## Среда и GPU

Устанавливаются зависимости проекта. Версии фактически установленной среды сохраняются рядом с результатами. При отсутствии CUDA запуск останавливается. Выбранный профиль использует **FP32 для модельных вычислений**; это исходный протокол сравнения, а не ускоренная BF16-оценка.


In [ ]:
run_logged([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt',
            '-r', 'requirements-neural.txt'], 'dependencies.log')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA недоступна. Выберите GPU runtime; CPU/BF16 автоматически не используются.')
properties = torch.cuda.get_device_properties(0)
free_bytes, total_bytes = torch.cuda.mem_get_info(0)
print('GPU:', properties.name)
print('Свободно/всего ГиБ:', round(free_bytes/2**30, 2), '/', round(total_bytes/2**30, 2))
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('Целевой GPU: 16 ГиБ при достаточной свободной памяти; успешный запуск не гарантируется.')
print('Пара Qwen FP32: около 12,3 ГБ (11,5 ГиБ) весов плюс активации и буферы; важна также RAM хоста.')
print('При OOM сохраните частичные результаты; точность и устройство автоматически не меняются.')
RUN_OUTPUT.mkdir(parents=True, exist_ok=True)
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(RUN_OUTPUT / 'colab-environment.txt').write_text(freeze, encoding='utf-8')
(RUN_OUTPUT / 'colab-runtime.json').write_text(json.dumps({
    'python': sys.version, 'torch': torch.__version__, 'cuda': torch.version.cuda,
    'gpu': properties.name, 'gpu_total_bytes': total_bytes,
    'precision': 'float32', 'profile': 'gpu-fp32', 'hf_home': os.environ['HF_HOME']
}, indent=2) + '\n', encoding='utf-8')


## Исходные данные и область пересчёта

Импортёр проверяет исходный ZIP и его части по SHA256. Экспорт сохраняет UUID, серии, варианты текста и происхождение записей. Исторические человеческие тексты остаются калибровочной справочной выборкой; они не превращаются в новый независимый тест.

`primary`: 1949 записей для детекторов. `quality`: 3449 записей для оценки качества. Одинаковые тексты могут оцениваться один раз с привязкой результата ко всем исходным записям. Частичные результаты не считаются полными.


In [ ]:
import hashlib
run_logged([sys.executable, 'scripts/import_source.py'], 'import-source.log')
for scope, expected in [('primary', 1949), ('quality', 3449)]:
    destination = RUN_OUTPUT / 'corpora' / scope
    if not destination.exists():
        run_logged([sys.executable, 'scripts/build_neural_corpus.py', '--scope', scope,
                    '--output', str(destination)], f'corpus-{scope}.log')
    manifest = json.loads((destination / 'manifest.json').read_text(encoding='utf-8'))
    corpus_path = destination / 'corpus.jsonl'
    actual_hash = hashlib.sha256(corpus_path.read_bytes()).hexdigest()
    if (manifest.get('scope') != scope or manifest.get('n_records') != expected
            or actual_hash != manifest.get('corpus_sha256')):
        raise ValueError(f'Корпус {scope} не соответствует ожидаемому манифесту. Не смешивайте разные версии запусков.')
    print(scope, ':', manifest['n_records'], 'записей;', manifest['n_unique_texts'], 'уникальных текстов; SHA256', actual_hash)


## Шесть публичных моделей с закреплёнными ревизиями

Загружаются только модели нейросетевого пересчёта: GPT‑2, D1, D2, D3 и пара Qwen‑1.5B для D4/D5. Экспериментальная модель генерации 0.5B здесь не используется. Все ревизии берутся из манифеста проекта; веса остаются вне репозитория и вне архива результатов.


In [ ]:
import re
entries = json.loads((REPO_ROOT / 'configs/neural/model_revisions.json').read_text(encoding='utf-8'))
by_id = {entry['model_id']: entry for entry in entries}
gpt2_job = json.loads((REPO_ROOT / 'configs/neural/gpt2-prefix.json').read_text(encoding='utf-8'))
required_models = [gpt2_job['model_id'], 'openai-community/roberta-base-openai-detector',
                   'Hello-SimpleAI/chatgpt-detector-roberta', 'desklib/ai-text-detector-v1.01',
                   'Qwen/Qwen2.5-1.5B', 'Qwen/Qwen2.5-1.5B-Instruct']
if len(set(required_models)) != 6:
    raise ValueError('Ожидаются шесть различных моделей нейросетевого пересчёта.')
for model_id in required_models:
    if model_id not in by_id or not re.fullmatch(r'[a-f0-9]{40}', by_id[model_id].get('revision', '')):
        raise ValueError('Модель не закреплена полным commit SHA: ' + model_id)
    print(model_id, by_id[model_id]['revision'])
arguments = [sys.executable, 'scripts/cache_neural_models.py', '--manifest', 'configs/neural/model_revisions.json']
for model_id in required_models:
    arguments.extend(['--model', model_id])
run_logged(arguments, 'cache-models.log')


## Последовательный пересчёт

Профиль `gpu-fp32` запускает модели последовательно, чтобы не загружать несколько тяжёлых заданий одновременно. Не запускайте эту ячейку в нескольких вкладках или процессах с одним каталогом результата.

Запуск может занять значительное время. Каждые 15 секунд отображается состояние журнала. После прерывания повторите эту же ячейку в той же среде; изменения кода или конфигурации требуют нового каталога результата. При ошибке сначала скачайте частичный архив.


In [ ]:
run_logged([sys.executable, 'scripts/run_neural_suite.py', '--profile', 'gpu-fp32',
            '--output', str(RUN_OUTPUT), '--allow-download'], 'neural-suite.log')
print('Команда завершилась. Проверьте manifests: только status=complete означает завершённый соответствующий этап.')


## Скачать полные или частичные результаты

Эту ячейку можно выполнять после успешного завершения, ошибки или ручной остановки. В архив входят результаты, контрольные точки и журналы; веса моделей исключены. Слово `partial` в имени архива используется консервативно, если общий манифест завершения не найден или не подтверждает `complete`. Наличие ZIP не подтверждает завершение научного эксперимента.


In [ ]:
from datetime import datetime, timezone
from google.colab import files
if not RUN_OUTPUT.exists():
    raise FileNotFoundError('Каталог результатов ещё не создан.')
def suite_download_status(directory):
    expected_jobs = {'gpt2-full', 'D1', 'D2', 'D3', 'D4-fp32'}
    candidate = Path(directory) / 'suite_manifest.json'
    try:
        manifest = json.loads(candidate.read_text(encoding='utf-8'))
    except (ValueError, OSError):
        return 'partial'
    if not isinstance(manifest, dict):
        return 'partial'
    jobs = manifest.get('jobs')
    if (manifest.get('status') == 'complete' and isinstance(jobs, dict)
            and set(jobs) == expected_jobs
            and all(isinstance(job, dict) and job.get('status') == 'complete'
                    for job in jobs.values())):
        return 'complete'
    return 'partial'

status = suite_download_status(RUN_OUTPUT)
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
archive_path = Path('/content') / f'SynSteg_Neural_FP32_{status}_{stamp}.zip'
excluded_suffixes = {'.safetensors', '.gguf', '.pt', '.pth'}
count = 0
with zipfile.ZipFile(archive_path, 'x', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for directory, prefix in [(RUN_OUTPUT, 'results'), (LOG_DIRECTORY, 'logs')]:
        if not directory.exists():
            continue
        for path in sorted(directory.rglob('*')):
            if (not path.is_file() or path.is_symlink()
                    or not path.resolve().is_relative_to(directory.resolve())
                    or path.suffix.lower() in excluded_suffixes
                    or path.name.startswith(('pytorch_model', 'model.safetensors'))
                    or any(part in {'synsteg-hf-cache', '.cache', 'models'} for part in path.relative_to(directory).parts)):
                continue
            archive.write(path, str(Path(prefix) / path.relative_to(directory)))
            count += 1
print('Архив:', archive_path.name, '; файлов:', count, '; размер МиБ:', round(archive_path.stat().st_size/2**20, 2))
print('Статус:', status, '— проверяйте манифесты каждого этапа перед интерпретацией результатов.')
files.download(str(archive_path))
